# DermaXplain - HAM10000 preprocessing and inventory notebook

This notebook consolidates dataset preprocessing and sanity checks for the DermaXplain practicum.

## Objectives

- validate local dataset structure
- clean non-image files from image/mask folders
- standardise mask naming conventions
- build dataset inventory tables
- verify image/mask matching for ISIC 2018 and HAM10000
- inspect image resolution and aspect-ratio distributions
- inspect lesion mask coverage distributions
- inspect HAM10000 class distribution
- visualise representative image/mask/overlay examples
- save clean inventory manifests for later notebooks

## Naming convention

Masks have been renamed to match image stems directly:

- image: `ISIC_xxxxxxx.jpg`
- mask: `ISIC_xxxxxxx.png`

In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import random

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

In [ ]:
ROOT = Path("..")

ISIC_DIR = ROOT / "data" / "isic2018"
ISIC_IMG_DIR = ISIC_DIR / "images"
ISIC_MASK_DIR = ISIC_DIR / "masks"

HAM_DIR = ROOT / "data" / "ham10000"
HAM_IMG_DIR = HAM_DIR / "images"
HAM_MASK_DIR = HAM_DIR / "masks"
HAM_META = HAM_DIR / "HAM10000_metadata.csv"

assert ISIC_IMG_DIR.exists(), f"Missing {ISIC_IMG_DIR}"
assert ISIC_MASK_DIR.exists(), f"Missing {ISIC_MASK_DIR}"
assert HAM_IMG_DIR.exists(), f"Missing {HAM_IMG_DIR}"
assert HAM_MASK_DIR.exists(), f"Missing {HAM_MASK_DIR}"
assert HAM_META.exists(), f"Missing {HAM_META}"

print("ROOT:", ROOT.resolve())
print("ISIC image dir:", ISIC_IMG_DIR.resolve())
print("ISIC mask dir :", ISIC_MASK_DIR.resolve())
print("HAM image dir :", HAM_IMG_DIR.resolve())
print("HAM mask dir  :", HAM_MASK_DIR.resolve())
print("HAM metadata  :", HAM_META.resolve())

## 1. Helper functions

In [ ]:
def suffix_count(dir_path: Path):
    files = [p for p in dir_path.iterdir() if p.is_file()]
    return Counter(p.suffix.lower() for p in files)

def list_files(dir_path: Path, suffix: str) -> list[Path]:
    return sorted([p for p in dir_path.iterdir() if p.is_file() and p.suffix.lower() == suffix])

def stems(dir_path: Path, suffix: str) -> set[str]:
    return {p.stem for p in list_files(dir_path, suffix)}

def load_rgb_u8(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

def load_mask_gray(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))

def load_mask_bin(path: Path) -> np.ndarray:
    m = load_mask_gray(path)
    return (m > 0).astype(np.uint8)

def image_hw(path: Path) -> tuple[int, int]:
    with Image.open(path) as im:
        w, h = im.size
    return h, w

def mask_coverage(mask_path: Path) -> float:
    m = load_mask_bin(mask_path)
    return float(m.mean())

def overlay_mask(img_rgb: np.ndarray, mask_bin: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    out = img_rgb.copy().astype(np.float32)
    red = np.zeros_like(out)
    red[..., 0] = 255
    lesion = mask_bin.astype(bool)
    out[lesion] = (1 - alpha) * out[lesion] + alpha * red[lesion]
    return out.astype(np.uint8)

## 2. Check file types before preprocessing

In [ ]:
print("ISIC images:", suffix_count(ISIC_IMG_DIR))
print("ISIC masks :", suffix_count(ISIC_MASK_DIR))

print("HAM images :", suffix_count(HAM_IMG_DIR))
print("HAM masks  :", suffix_count(HAM_MASK_DIR))

## 3. Move `.txt` files out of image/mask folders

This keeps image and mask directories clean and avoids file-scanning issues later.

In [ ]:
# Only to run once to move txt files to a separate directory (if not done already)

def move_txt_files(base_dir: Path):
    moved = []

    for sub in ["images", "masks"]:
        src_dir = base_dir / sub
        if not src_dir.exists():
            continue

        dest_dir = base_dir / "txt" / sub
        dest_dir.mkdir(parents=True, exist_ok=True)

        for p in src_dir.iterdir():
            if p.is_file() and p.suffix.lower() == ".txt":
                dest = dest_dir / p.name
                if not dest.exists():
                    shutil.move(str(p), str(dest))
                    moved.append((str(p), str(dest)))

    return moved

moved_isic = move_txt_files(ISIC_DIR)
moved_ham = move_txt_files(HAM_DIR)

print(f"ISIC moved txt files: {len(moved_isic)}")
print(f"HAM moved txt files : {len(moved_ham)}")

In [ ]:
# Check that the moved files are indeed txt files and that they are no longer in the original directories
print("After cleanup:")
print("ISIC images:", suffix_count(ISIC_IMG_DIR))
print("ISIC masks :", suffix_count(ISIC_MASK_DIR))
print("HAM images :", suffix_count(HAM_IMG_DIR))
print("HAM masks  :", suffix_count(HAM_MASK_DIR))

## 4. Standardize mask names

Masks were originally named like:

- `ISIC_xxxxxxx_segmentation.png`

They are renamed to:

- `ISIC_xxxxxxx.png`

This simplifies image/mask mapping.

In [ ]:
def remove_segmentation_suffix(mask_dir: Path, dry_run: bool = False) -> list[tuple[str, str]]:
    renames = []

    for p in mask_dir.iterdir():
        if not p.is_file() or p.suffix.lower() != ".png":
            continue
        if not p.stem.endswith("_segmentation"):
            continue

        new_name = p.stem.removesuffix("_segmentation") + ".png"
        new_path = p.with_name(new_name)

        if new_path.exists():
            raise FileExistsError(f"Collision detected: {new_path}")

        renames.append((p.name, new_name))

        if not dry_run:
            p.rename(new_path)

    return renames

In [ ]:
# Run only if needed
# renames_isic = remove_segmentation_suffix(ISIC_MASK_DIR, dry_run=False)
# renames_ham  = remove_segmentation_suffix(HAM_MASK_DIR, dry_run=False)

# print("ISIC renamed:", len(renames_isic))
# print("HAM renamed :", len(renames_ham))

print("Mask naming convention is assumed to be already standardized.")

## 5. Dataset inventory and stem matching

In [ ]:
# ISIC
isic_img_stems = stems(ISIC_IMG_DIR, ".jpg")
isic_mask_stems = stems(ISIC_MASK_DIR, ".png")

isic_pairs = isic_img_stems & isic_mask_stems
isic_img_only = sorted(isic_img_stems - isic_mask_stems)
isic_mask_only = sorted(isic_mask_stems - isic_img_stems)

# HAM
ham_img_stems = stems(HAM_IMG_DIR, ".jpg")
ham_mask_stems = stems(HAM_MASK_DIR, ".png")

ham_pairs = ham_img_stems & ham_mask_stems
ham_img_only = sorted(ham_img_stems - ham_mask_stems)
ham_mask_only = sorted(ham_mask_stems - ham_img_stems)

inventory_summary = pd.DataFrame([
    {
        "dataset": "ISIC 2018",
        "images": len(isic_img_stems),
        "masks": len(isic_mask_stems),
        "matched_pairs": len(isic_pairs),
        "image_only": len(isic_img_only),
        "mask_only": len(isic_mask_only)
    },
    {
        "dataset": "HAM10000",
        "images": len(ham_img_stems),
        "masks": len(ham_mask_stems),
        "matched_pairs": len(ham_pairs),
        "image_only": len(ham_img_only),
        "mask_only": len(ham_mask_only)
    }
])

display(inventory_summary)

print("ISIC images without mask:", len(isic_img_only))
print("ISIC masks without image:", len(isic_mask_only))

print("HAM images without mask:", len(ham_img_only))
print("HAM masks without image:", len(ham_mask_only))

display(pd.DataFrame({"isic_img_only": pd.Series(isic_img_only)}).head(10))
display(pd.DataFrame({"ham_img_only": pd.Series(ham_img_only)}).head(10))

## 6. Image Resolution statistics

In [ ]:
def resolution_df(img_dir: Path, suffix: str) -> pd.DataFrame:
    paths = list_files(img_dir, suffix)

    rows = []
    for p in paths:
        h, w = image_hw(p)
        rows.append({
            "stem": p.stem,
            "height": h,
            "width": w,
            "pixels": h * w,
            "aspect_ratio": w / h
        })
    return pd.DataFrame(rows)

isic_res = resolution_df(ISIC_IMG_DIR, ".jpg")
ham_res = resolution_df(HAM_IMG_DIR, ".jpg")

print("ISIC resolution summary:")
display(isic_res[["height", "width", "pixels", "aspect_ratio"]].describe())

print("HAM resolution summary:")
display(ham_res[["height", "width", "pixels", "aspect_ratio"]].describe())

# Plots of pixel count distributions
plt.figure(figsize=(7, 4))
plt.hist(isic_res["pixels"], bins=30)
plt.title("ISIC 2018 image pixel count distribution")
plt.xlabel("pixels")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(ham_res["pixels"], bins=30)
plt.title("HAM10000 image pixel count distribution")
plt.xlabel("pixels")
plt.ylabel("count")
plt.tight_layout()
plt.show()


# Plots of aspect ratio distributions
plt.figure(figsize=(7, 4))
plt.hist(isic_res["aspect_ratio"], bins=30)
plt.title("ISIC 2018 image aspect ratio distribution")
plt.xlabel("aspect ratio")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(ham_res["aspect_ratio"], bins=30)
plt.title("HAM10000 image aspect ratio distribution")
plt.xlabel("aspect ratio")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 7. Image-mask size consistency

Matching stems are not enough. We also need matching dimensions.

In [ ]:
def pair_size_check(img_dir: Path, mask_dir: Path, img_suffix=".jpg", mask_suffix=".png") -> pd.DataFrame:
    common = sorted(stems(img_dir, img_suffix) & stems(mask_dir, mask_suffix))
    rows = []

    for stem in common:
        img_path = img_dir / f"{stem}{img_suffix}"
        mask_path = mask_dir / f"{stem}{mask_suffix}"

        img_h, img_w = image_hw(img_path)
        mask_h, mask_w = image_hw(mask_path)

        rows.append({
            "stem": stem,
            "img_h": img_h,
            "img_w": img_w,
            "mask_h": mask_h,
            "mask_w": mask_w,
            "same_size": (img_h == mask_h) and (img_w == mask_w)
        })

    return pd.DataFrame(rows)

isic_pair_sizes = pair_size_check(ISIC_IMG_DIR, ISIC_MASK_DIR)
ham_pair_sizes = pair_size_check(HAM_IMG_DIR, HAM_MASK_DIR)

print("ISIC same-size counts:")
display(isic_pair_sizes["same_size"].value_counts())

print("HAM same-size counts:")
display(ham_pair_sizes["same_size"].value_counts())

display(isic_pair_sizes.loc[~isic_pair_sizes["same_size"]].head(10))
display(ham_pair_sizes.loc[~ham_pair_sizes["same_size"]].head(10))

## 8. Mask coverage statistics

Mask coverage = fraction of image pixels that belong to the lesion.
This is useful later for XAI analysis and pseudo-concept generation.

In [ ]:
def coverage_df(mask_dir: Path, suffix: str) -> pd.DataFrame:
    paths = list_files(mask_dir, suffix)
    rows = []

    for p in paths:
        rows.append({
            "stem": p.stem,
            "mask_coverage": mask_coverage(p)
        })

    return pd.DataFrame(rows)

isic_cov = coverage_df(ISIC_MASK_DIR, ".png")
ham_cov = coverage_df(HAM_MASK_DIR, ".png")

print("ISIC mask coverage summary:")
display(isic_cov["mask_coverage"].describe())

print("HAM mask coverage summary:")
display(ham_cov["mask_coverage"].describe())

## Plots of mask coverage distributions
plt.figure(figsize=(7, 4))
plt.hist(isic_cov["mask_coverage"], bins=30)
plt.title("ISIC 2018 lesion mask coverage")
plt.xlabel("lesion area fraction")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(ham_cov["mask_coverage"], bins=30)
plt.title("HAM10000 lesion mask coverage")
plt.xlabel("lesion area fraction")
plt.ylabel("count")
plt.tight_layout()
plt.show()

In [ ]:
plt.boxplot(
    [isic_cov.mask_coverage, ham_cov.mask_coverage],
    labels=["ISIC 2018", "HAM10000"]
)
plt.ylabel("Lesion area fraction")
plt.title("Lesion coverage distribution")
plt.show()

## 9. HAM10000 metadata and class distribution

In [ ]:
# Load and inspect HAM10000 metadata

ham_meta = pd.read_csv(HAM_META)

print(ham_meta.shape)
ham_meta.head()
ham_meta.info()
ham_meta["dx"].value_counts()



In [ ]:
# Binary target mapping
ham_meta["label"] = ham_meta["dx"].apply(lambda x: 1 if x == "mel" else 0)
ham_meta["label_name"] = ham_meta["dx"].apply(lambda x: "malignant" if x == "mel" else "benign")
ham_meta[["image_id", "dx", "label", "label_name"]].head()

# Inspect class distribution
class_counts = ham_meta["label_name"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values, color=["blue", "orange"])
plt.title("Class Distribution in HAM10000")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()  

## 10. Additional mask diagnostics

We add two preprocessing diagnostics that may affect later XAI evaluation:

- **border-touching lesions**: masks that touch one or more image borders
- **tiny lesions**: masks with very small area fraction

These cases are not removed here. They are flagged in the inventory so they can be analysed later.

In [ ]:
def border_touch_info(mask_path: Path) -> dict:
    """
    Detect whether the lesion mask touches the image borders.
    """
    mask = load_mask_bin(mask_path)

    top = bool(mask[0, :].any())
    bottom = bool(mask[-1, :].any())
    left = bool(mask[:, 0].any())
    right = bool(mask[:, -1].any())

    n_borders_touched = sum([top, bottom, left, right])

    return {
        "stem": mask_path.stem,
        "touch_top": top,
        "touch_bottom": bottom,
        "touch_left": left,
        "touch_right": right,
        "touches_any_border": n_borders_touched > 0,
        "n_borders_touched": n_borders_touched
    }


def border_touch_df(mask_dir: Path, suffix=".png") -> pd.DataFrame:
    rows = []
    for mask_path in list_files(mask_dir, suffix):
        rows.append(border_touch_info(mask_path))
    return pd.DataFrame(rows)


def classify_mask_size(coverage: float) -> str:
    """
    Classify lesion size based on mask coverage fraction.
    """
    if coverage < 0.01:
        return "very_tiny"
    elif coverage < 0.02:
        return "tiny"
    elif coverage < 0.05:
        return "small"
    else:
        return "normal"

In [ ]:
isic_border = border_touch_df(ISIC_MASK_DIR)
ham_border = border_touch_df(HAM_MASK_DIR)

print("ISIC border-touch summary:")
display(isic_border["touches_any_border"].value_counts())
display(isic_border["n_borders_touched"].value_counts().sort_index())

print("HAM border-touch summary:")
display(ham_border["touches_any_border"].value_counts())
display(ham_border["n_borders_touched"].value_counts().sort_index())


isic_cov["mask_size_class"] = isic_cov["mask_coverage"].apply(classify_mask_size)
ham_cov["mask_size_class"] = ham_cov["mask_coverage"].apply(classify_mask_size)

print("ISIC mask size classes:")
display(isic_cov["mask_size_class"].value_counts())

print("HAM mask size classes:")
display(ham_cov["mask_size_class"].value_counts())

isic_diag = isic_cov.merge(isic_border, on="stem", how="left")
ham_diag = ham_cov.merge(ham_border, on="stem", how="left")

display(isic_diag.head())
display(ham_diag.head())

print("ISIC suspicious cases:")
isic_suspicious = isic_diag.loc[
    (isic_diag["touches_any_border"]) | (isic_diag["mask_coverage"] < 0.02)
].copy()
display(isic_suspicious.head(20))
print("Count:", len(isic_suspicious))

print("HAM suspicious cases:")
ham_suspicious = ham_diag.loc[
    (ham_diag["touches_any_border"]) | (ham_diag["mask_coverage"] < 0.02)
].copy()
display(ham_suspicious.head(20))
print("Count:", len(ham_suspicious))




## 11. Build inventory manifests


In [ ]:
def build_inventory(img_dir: Path, mask_dir: Path, img_suffix=".jpg", mask_suffix=".png") -> pd.DataFrame:
    img_stems = stems(img_dir, img_suffix)
    mask_stems = stems(mask_dir, mask_suffix)
    all_stems = sorted(img_stems | mask_stems)

    rows = []
    for stem in all_stems:
        img_exists = stem in img_stems
        mask_exists = stem in mask_stems

        row = {
            "stem": stem,
            "image_exists": img_exists,
            "mask_exists": mask_exists
        }

        if img_exists:
            img_path = img_dir / f"{stem}{img_suffix}"
            img_h, img_w = image_hw(img_path)
            row["img_h"] = img_h
            row["img_w"] = img_w
            row["pixels"] = img_h * img_w
            row["aspect_ratio"] = img_w / img_h
        else:
            row["img_h"] = np.nan
            row["img_w"] = np.nan
            row["pixels"] = np.nan
            row["aspect_ratio"] = np.nan

        if mask_exists:
            mask_path = mask_dir / f"{stem}{mask_suffix}"
            mask_h, mask_w = image_hw(mask_path)
            cov = mask_coverage(mask_path)

            row["mask_h"] = mask_h
            row["mask_w"] = mask_w
            row["mask_coverage"] = cov
            row["mask_pixels"] = int(round(cov * mask_h * mask_w))
            row["mask_size_class"] = classify_mask_size(cov)
        else:
            row["mask_h"] = np.nan
            row["mask_w"] = np.nan
            row["mask_coverage"] = np.nan
            row["mask_pixels"] = np.nan
            row["mask_size_class"] = np.nan

        row["same_size"] = (
            img_exists and mask_exists and
            row["img_h"] == row["mask_h"] and
            row["img_w"] == row["mask_w"]
        )

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
isic_inventory = build_inventory(ISIC_IMG_DIR, ISIC_MASK_DIR)
ham_inventory = build_inventory(HAM_IMG_DIR, HAM_MASK_DIR)

isic_inventory = isic_inventory.merge(
    isic_border[[
        "stem",
        "touch_top",
        "touch_bottom",
        "touch_left",
        "touch_right",
        "touches_any_border",
        "n_borders_touched"
    ]],
    on="stem",
    how="left"
)

ham_inventory = ham_inventory.merge(
    ham_border[[
        "stem",
        "touch_top",
        "touch_bottom",
        "touch_left",
        "touch_right",
        "touches_any_border",
        "n_borders_touched"
    ]],
    on="stem",
    how="left"
)

display(isic_inventory.head())
display(ham_inventory.head())

In [ ]:
ham_inventory = ham_inventory.merge(
    ham_meta[["image_id", "dx", "label", "label_name"]],
    left_on="stem",
    right_on="image_id",
    how="left"
).drop(columns=["image_id"])

display(ham_inventory.head())

In [ ]:
def inventory_diagnostic_summary(df: pd.DataFrame, name: str):
    print(f"\n{name} - same_size")
    display(df["same_size"].value_counts())

    print(f"{name} - mask_size_class")
    display(df["mask_size_class"].value_counts())

    print(f"{name} - touches_any_border")
    display(df["touches_any_border"].value_counts())

    print(f"{name} - n_borders_touched")
    display(df["n_borders_touched"].value_counts().sort_index())

    print(f"{name} - border-touch rate")
    print(f"{100 * df['touches_any_border'].mean():.2f}%")

inventory_diagnostic_summary(isic_inventory, "ISIC")
inventory_diagnostic_summary(ham_inventory, "HAM")

In [ ]:
print("ISIC border-touch percentage:")
print(isic_inventory["touches_any_border"].mean())

print("HAM border-touch percentage:")
print(ham_inventory["touches_any_border"].mean())

## 12. Gallery

In [ ]:
def show_triplets(stems_list, img_dir: Path, mask_dir: Path, n=6, title="Gallery"):
    chosen = stems_list[:n]
    fig, axes = plt.subplots(len(chosen), 3, figsize=(11, 4 * len(chosen)))

    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, stem in enumerate(chosen):
        img = load_rgb_u8(img_dir / f"{stem}.jpg")
        mask = load_mask_bin(mask_dir / f"{stem}.png")
        overlay = overlay_mask(img, mask, alpha=0.4)

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{stem} - image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(f"mask (coverage={mask.mean():.3f})")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("overlay")
        axes[i, 2].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# random examples from HAM
random_ham_stems = random.sample(sorted(list(ham_pairs)), min(6, len(ham_pairs)))
show_triplets(random_ham_stems, HAM_IMG_DIR, HAM_MASK_DIR, n=len(random_ham_stems), title="HAM10000 random examples")

In [ ]:
# smallest lesions in HAM
smallest_ham = (
    ham_inventory.loc[ham_inventory["mask_exists"]]
    .sort_values("mask_coverage", ascending=True)["stem"]
    .tolist()
)

show_triplets(smallest_ham, HAM_IMG_DIR, HAM_MASK_DIR, n=min(6, len(smallest_ham)), title="HAM10000 smallest lesion masks")

In [ ]:
# largest lesions in HAM
largest_ham = (
    ham_inventory.loc[ham_inventory["mask_exists"]]
    .sort_values("mask_coverage", ascending=False)["stem"]
    .tolist()
)

show_triplets(largest_ham, HAM_IMG_DIR, HAM_MASK_DIR, n=min(6, len(largest_ham)), title="HAM10000 largest lesion masks")

## 12. Save clean invenmtory manifests

In [ ]:
OUT_DIR = ROOT / "data" / "preprocessed_manifests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

isic_inventory_path = OUT_DIR / "isic2018_inventory.csv"
ham_inventory_path = OUT_DIR / "ham10000_inventory.csv"

isic_inventory.to_csv(isic_inventory_path, index=False)
ham_inventory.to_csv(ham_inventory_path, index=False)

print("Saved:", isic_inventory_path.resolve())
print("Saved:", ham_inventory_path.resolve())